# Phase 5: LightGCN — Graph-Powered Bias Amplification

This notebook introduces **LightGCN** (He et al., 2020) as the first graph-based collaborative filtering model in our longitudinal closed-loop recommender bias study.

### Core Theoretical Focus:
1. **Graph Convolution Mechanics**: How normalized bipartite graph propagation ($E^{(l+1)} = D^{-1/2} A D^{-1/2} E^{(l)}$) aggregates high-order neighbor signals.
2. **The Graph Centrality Hypothesis**: Unlike Matrix Factorization (MF) and Bayesian Personalized Ranking (BPR)—which amplify bias purely through click gradients and sampling frequencies—LightGCN possesses a **structural bias mechanism** rooted in bipartite degree centrality.
3. **Multi-Tier Benchmark Trajectories**: Tracking Catalog Exposure Inequality ($G$), User Taste Drift ($D_{\text{KL}}$), Recommendation Accuracy ($NDCG@10$), and Manifold Effective Rank ($S_{\text{eff}}$) across 10 generations.
4. **Manifold & Quantitative Dynamics**: Visualizing 2D t-SNE attractor consolidation, genre cluster blurring, kernel density singularity, and 50-generation dynamic animations.

In [1]:
import os
import sys
import json
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.manifold import TSNE
from IPython.display import Image as IPImage, display

sys.path.insert(0, '..')

from src.data.dataset import load_movielens_1m
from src.data.buffer import merge_and_deduplicate
from src.adapters.lightgcn_adapter import LightGCNAdapter
from src.engine import simulate_user_clicks
from src.metrics.macro import compute_exposure_gini
from src.metrics.geometry import (
    compute_reflection_corrected_procrustes,
    compute_effective_rank,
    compute_drift_metrics_32d,
)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150
})


## 1. Multi-Tier Benchmark Metric Trajectories

We evaluate 4 model architectures across 10 feedback generations on MovieLens-1M:
- **Matrix Factorization (MF)**: Pointwise MSE baseline
- **Bayesian Personalized Ranking (BPR)**: Pairwise ranking baseline
- **Most Popular (MostPop)**: Non-personalized heuristic ceiling
- **LightGCN (Graph CF)**: 3-layer bipartite graph convolution network

### Key Hypotheses:
- **Gini ($G$)**: LightGCN compounds inequality faster than BPR because high-degree items receive larger gradient allocations via shared neighbors.
- **Taste Drift ($D_{\text{KL}}$)**: Users converge toward popular genres faster as multi-hop message passing homogenizes latent representations.
- **NDCG@10**: LightGCN exhibits superior initial ranking accuracy at $t=0$, but degrades faster under feedback loop concentration.
- **Effective Rank ($S_{\text{eff}}$)**: Latent dimensionality collapses into fewer active principal components due to degree centrality dominance.

In [2]:
bench_img_path = '../docs/images/phase_5_lightgcn/benchmark_trajectories_lightgcn.png'
if os.path.exists(bench_img_path):
    display(IPImage(filename=bench_img_path, width=950))
else:
    print('Trajectory asset not found at', bench_img_path)


## 2. Ingest Dataset & Multi-Hop Graph Convolution Pipeline

We load MovieLens-1M (threshold $\ge 4.0$, 80/20 train/test split) and extract primary movie genres from `movies.dat`.

In [3]:
train_mat, test_mat, u2c, i2c = load_movielens_1m('../ml-1m/ratings.dat', threshold=4.0, split_ratio=0.8)
n_users, n_items = train_mat.shape

df_movies = pd.read_csv('../ml-1m/movies.dat', sep='::', names=['item_id', 'title', 'genres'], engine='python', encoding='latin-1')
primary_genres = ['Unknown'] * n_items
for _, row in df_movies.iterrows():
    raw_id = row['item_id']
    if raw_id in i2c:
        code = i2c[raw_id]
        genres = str(row['genres']).split('|')
        primary_genres[code] = genres[0] if genres else 'Unknown'

print(f"Ingestion complete: {n_users} users, {n_items} items, {train_mat.nnz} implicit feedback interactions.")


## 3. Joint 2D t-SNE Manifold Projection

To analyze latent manifold deformation without artificial coordinate system shifts, we apply **Reflection-Corrected Orthogonal Procrustes** alignment on $V_t$ relative to $V_0$, followed by a **joint t-SNE embedding** on $[V_0; V_{\text{final}}]$.

## 4. Visual 1: Popularity Gravity Attractors & Drift Vectors

The 2D t-SNE projection maps all 3,533 catalog items at Generation $t=0$ (left) and Generation $t=9$ (right). Red vector arrows trace the displacement trajectories of the top 50 most viral items.

In [4]:
pop_drift_path = '../docs/images/phase_5_lightgcn/tsne_popularity_drift.png'
if os.path.exists(pop_drift_path):
    display(IPImage(filename=pop_drift_path, width=950))
else:
    print('Asset not found at', pop_drift_path)


## 5. Visual 2: Primary Movie Genre Cluster Blurring

How does graph propagation affect genre identity? In LightGCN, items that share co-consumption edges blend their representations across multiple hops. As the feedback loop progresses, genre boundaries soften in favor of a monolithic "popularity super-cluster."

In [5]:
genre_img_path = '../docs/images/phase_5_lightgcn/tsne_genre_clusters.png'
if os.path.exists(genre_img_path):
    display(IPImage(filename=genre_img_path, width=950))
else:
    print('Asset not found at', genre_img_path)


## 6. Visual 3: Latent Space Density Compression (KDE)

2D Kernel Density Estimation (KDE) contour mapping comparing factor space dispersion at $t=0$ (blue) versus $t=9$ (green). LightGCN displays the most extreme density singularity among all studied algorithms.

In [6]:
density_img_path = '../docs/images/phase_5_lightgcn/tsne_manifold_density.png'
if os.path.exists(density_img_path):
    display(IPImage(filename=density_img_path, width=950))
else:
    print('Asset not found at', density_img_path)


## 7. Dynamic 50-Generation Latent Trajectory Animations

Below are two synchronized 50-frame animated sequences capturing continuous closed-loop bias propagation over 50 generations ($t=0 \to 49$).

### Animation 1: Dynamic Manifold Drift alongside Live Gini & Effective Rank
- **Left**: Continuous 2D manifold trajectory with red tracking paths for the top 30 viral movies.
- **Top Right**: Live tracking of Catalog Exposure Gini ($G$).
- **Bottom Right**: Live tracking of Latent Manifold Effective Rank ($S_{\text{eff}}$).

In [7]:
gif1_path = '../docs/images/phase_5_lightgcn/embedding_drift_trajectory.gif'
if os.path.exists(gif1_path):
    display(IPImage(filename=gif1_path, width=900))
else:
    print('Animation asset not found at', gif1_path)


### Animation 2: 32-D Quantitative Metric Trajectories across Popularity Tiers
Tracks exact mathematical measurements computed directly in $\mathbb{R}^{32}$ across:
- **Short-tail items** ($\ge 250$ interactions, orange)
- **Mid-tail items** ($50 - 249$ interactions, purple)
- **Long-tail items** ($< 50$ interactions, green)

Panels monitor:
1. **Mean Euclidean Drift ($\bar{d}_{\text{Euc}}$)**: True straight-line physical displacement after Procrustes alignment.
2. **Mean Cosine Drift ($\bar{d}_{\text{Cos}}$)**: Pure angular rotation ($1 - \cos \theta$).
3. **Mean Vector L2 Norm ($\bar{\|\mathbf{v}\|}_2$)**: Vector magnitude expansion.

In [8]:
gif2_path = '../docs/images/phase_5_lightgcn/quantitative_embedding_drift_trajectory.gif'
if os.path.exists(gif2_path):
    display(IPImage(filename=gif2_path, width=900))
else:
    print('Animation asset not found at', gif2_path)


## 8. Theoretical Synthesis & The Graph Centrality Hypothesis

### Summary of Findings:
1. **Graph Structural Amplification**: LightGCN's $D^{-1/2} A D^{-1/2}$ normalisation allocates stronger message-passing weights to high-degree items. Popular items act as geometric attractors, pulling neighboring embeddings toward them in latent space.
2. **The High-Accuracy Paradox**: LightGCN achieves higher initial accuracy ($NDCG@10$) at generation $t=0$, but compounds inequality faster than Matrix Factorization and BPR.
3. **Structural Debiasing Requirement**: Non-graph interventions (such as inverse propensity weighting on clicks alone) are insufficient for graph models; debiasing must operate directly on the graph adjacency normalisation.